<a href="https://colab.research.google.com/github/AmisiJospin/AfriLanguageMalawi/blob/main/notebooks/afrispeech_selector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Get AfriSpeech data into your training notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfriSpeech/afrispeech-selector/blob/main/notebooks/afrispeech_selector.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/AfriSpeech/afrispeech-selector/blob/main/notebooks/afrispeech_selector.ipynb)

**Purpose:** drop-in cells for your own training notebook (Colab or Kaggle). They
install AfriSpeech Selector and pull your selected language(s) straight into the
running session, so your training script has the data without manual downloads.

It's clean read speech with aligned transcripts — a natural fit for **TTS**
(Option A). It also works for **ASR** (Option B), handy for supplementing
low-resource languages.

Use whatever runtime your **training** needs (a GPU for the actual model). On
**Kaggle**, turn **Internet ON** (`Settings → Internet`) so the data can download.

## Install

> **Kaggle:** turn **Internet ON** first — right sidebar → **Settings → Internet → On**
> (needs a phone-verified account). Without it the clone fails with
> `Could not resolve host: github.com`. Colab has internet on by default.

In [1]:
# Clone and install (editable).
!git clone -q https://github.com/AfriSpeech/afrispeech-selector.git
%cd afrispeech-selector
!pip install -q -e .

/content/afrispeech-selector
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for afrispeech-selector (pyproject.toml) ... done


## (Optional) Hugging Face token — faster downloads
Anonymous downloads are rate-limited (slower). A free **read** token
(https://huggingface.co/settings/tokens) lifts the limit. Run this and paste one,
or skip it to stay anonymous.

In [ ]:
import os, getpass
_tok = getpass.getpass("HF token (optional — press Enter to skip): ").strip()
if _tok:
    os.environ["HF_TOKEN"] = _tok
    print("Token set — higher rate limits / faster downloads.")
else:
    print("No token — downloads still work, just slower (rate-limited).")

## Option A — a local set for a TTS framework
Writes `wavs/` + a manifest into the session for Piper / VITS / MeloTTS / LJSpeech;
point your TTS data-prep at the folder. Transcripts are written verbatim.

In [ ]:
# Invoked as a module (`python -m …`) so it works regardless of PATH in notebooks.
# --yes proceeds without prompting if a language has fewer hours than requested.
!python -m afrispeech_selector --languages twi_twi --total-hours 5 --out data/twi --format ljspeech -y
!ls data/twi && head -2 data/twi/metadata.csv
# swap --format for piper / vits / melo (or combine: --format vits,melo)

### Preview a few clips (audio + text)
Sanity-check the download — play a few WAVs and read their transcripts.

In [ ]:
# Preview a few downloaded clips — listen and read the transcript
import random
from pathlib import Path
from IPython.display import Audio, display

base = Path("data/twi")
rows = [ln.split("|") for ln in (base / "metadata.csv").read_text(encoding="utf-8").splitlines() if ln]
for uid, text, *_ in random.sample(rows, min(3, len(rows))):
    print(f"\n[{uid}]  {text}")
    display(Audio(str(base / "wavs" / f"{uid}.wav")))

## Option B — stream into ASR training (no copy)
Well suited to supplementing low-resource ASR. Hand the `IterableDataset` to your
🤗 `Trainer` / DataLoader — nothing is written to disk.

In [ ]:
from afrispeech_selector import stream_dataset

ds = stream_dataset(
    ["twi_twi"],                 # one or more languages (see afrispeech-select --list-langs)
    split="train",
    max_seconds=5 * 3600,         # ~5 h; or per_language=200 for a clip count
    target_sampling_rate=16000,   # min/max clip length default to 3-15s
)

# `ds` is a datasets.IterableDataset — plug it into your training loop:
for batch in ds.iter(batch_size=8):
    audio, text = batch["audio"], batch["text"]
    break  # <-- replace with your forward / loss step
print("streaming OK:", text[0][:80])

### ASR with a cached dataset (`load_from_disk`)
If you'd rather have random access (multiple epochs, shuffling) than a stream.

In [ ]:
!python -m afrispeech_selector --languages twi_twi --total-hours 5 --out data/twi --format disk --target-sr 16000 -y
from datasets import load_from_disk
ds = load_from_disk("data/twi").train_test_split(test_size=0.1)
print(ds)

### More than one language
Pass several names, or select across the dataset by strength — the budget is split
evenly per language:

```python
stream_dataset(["twi_twi", "ewe_ewe", "ga_gaa"], split="train", max_seconds=2 * 3600)
```
```bash
!python -m afrispeech_selector --top 10 --max-per-country 2 --total-hours 24 --out data/multi --format ljspeech -y
```

Output defaults to `data/<language>` if you omit `--out`. Full options and the
language list: https://github.com/AfriSpeech/afrispeech-selector